# Séance 12 — Nouvelle typologie, réentraînement et promotion
*Session 12 — New fraud pattern, retraining and promotion*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/12_nouvelle_fraude_reentrainement.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ○ démonstration |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | — hors parcours |

**À la fin de cette séance, vous saurez :**

1. diagnostiquer pourquoi un modèle « ne voit pas » une nouvelle fraude (angle mort, pas dérive) ;
2. comparer une mitigation immédiate (règle experte) et une réponse structurelle (nouvelles variables) ;
3. appliquer une grille de promotion champion/challenger ;
4. expliquer pourquoi un meilleur PR-AUC ne suffit pas toujours à promouvoir.

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

**Scénario.** Quinze jours après la mise en production apparaît l'**arnaque au virement instantané** : un fraudeur manipule
le membre, qui fait lui-même un virement Interac vers un nouveau bénéficiaire, depuis son appareil habituel. Rien d'anormal pour
les variables de la carte… Le pipeline : injecte la période (étape 10), constate, mitige par une règle (R11), crée 6 variables de
virement (jeu v2), réentraîne et teste le challenger contre le champion (étape 11).

In [ ]:
cfg = preparer("reentrainer")
inc = lire_json(cfg.chemin("artefacts") / "incidents" / "rapport_incident.json")
diag = inc["diagnostic"]
print("Détection — typologies connues :", diag["taux_detection_typologies_connues_pct"], "% | nouvelle :", diag["taux_detection_nouvelle_pct"], "%")
print("Cause racine :", diag["cause_racine"])
pd.DataFrame(diag["detail_par_typologie"])

### Exercice 12.1 — 🟢 Tous les parcours

À partir de la table des décisions (`table("gold", "decisions")`), recalculez `detection`, un `DataFrame` indexé
par `typologie_fraude` (fraudes seulement) avec `cas` (nombre), `detectees` (somme de `alerte`) et `taux_detection_pct`
(arrondi à 0,1).

In [ ]:
# VOTRE CODE ICI
dec = table("gold", "decisions")
detection = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_ref = pd.DataFrame(diag["detail_par_typologie"]).set_index("typologie_fraude")["taux_detection_pct"]
verifier(np.allclose(detection.loc[_ref.index, "taux_detection_pct"].astype(float), _ref.astype(float)),
         "Mêmes taux de détection que le rapport d'incident")

In [ ]:
print("Mitigation immédiate (règle R11) :", inc["mitigation_immediate"])
print("Nouvelles variables (v2) :", inc["nouvelles_variables"])

In [ ]:
reent = lire_json(cfg.chemin("artefacts") / "modele" / "rapport_reentrainement.json")
{k: reent[k] for k in ["pr_auc_champion_meme_test", "pr_auc_challenger_meme_test", "gain_pr_auc",
                       "taux_detection_champion", "taux_detection_challenger",
                       "cout_par_1000tx_champion", "cout_par_1000tx_challenger", "promotion"]}

### Exercice 12.2 — 🟣 Gestion + Informatique

Écrivez `promouvoir(r, rt)` qui applique la grille du pipeline et renvoie `(decision, conditions)` :
`test_ab_favorable` (`r["rapport_ab"]["decision"] == "DEPLOYER_B"`), `gain_pr_auc_suffisant` (`r["gain_pr_auc"] >= rt["gain_minimal_pr_auc"]`),
`pas_de_regression_rappel` (rappel du challenger ≥ rappel du champion − `rt["garde_fou_baisse_rappel"]`). La décision est vraie si
**toutes** les conditions le sont.

In [ ]:
# VOTRE CODE ICI
def promouvoir(r, rt):
    ...

decision, conditions = promouvoir(reent, cfg["reentrainement"])
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(decision == reent["promotion"] and conditions == reent["conditions_de_promotion"], "Même verdict que le pipeline")

### Exercice 12.3 — 🔷 Informatique — approfondissement

Les nouvelles variables séparent-elles la nouvelle typologie ? Dans `table("gold", "caracteristiques")`, comparez la
moyenne de chaque variable de `inc["nouvelles_variables"]` entre l'arnaque au virement (`typologie_fraude == cfg["nouvelle_fraude"]["nom"]`)
et les transactions légitimes. Stockez le tableau dans `comparaison` (colonnes `arnaque`, `legitime`).

In [ ]:
# VOTRE CODE ICI
comparaison = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(list(comparaison.columns) == ["arnaque", "legitime"] and len(comparaison) >= 3)
verifier((comparaison.arnaque > comparaison.legitime).sum() >= len(comparaison) // 2, "Les variables v2 s'écartent de la normale pour l'arnaque")

**❓ Question (🔶 Gestion — décision d'affaires) —** Le challenger a un meilleur PR-AUC mais n'est pas promu. Expliquez ce verdict à un vice-président en trois phrases. Que proposeriez-vous pour la suite ?

*Votre réponse ici.*

## Bilan

- Une nouvelle fraude est souvent un angle mort (information absente), pas une dérive.
- Règle experte immédiate + nouvelles variables + réentraînement : trois horizons de réponse.
- La promotion suit une grille explicite; refuser un modèle « meilleur » est un résultat légitime.

**Pour aller plus loin :**

- `src/fraude360/newfraud/` et `src/fraude360/retraining/reentrainement.py`.

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*